# SubSight SegFormer chrono (Kaggle)

Setup once: Settings sidebar > Accelerator = GPU, Internet = ON. Then Run All.
Same protocol as Colab: merged Mini set, chrono 60/20/20, seed 42, 20 epochs, 256px.
Artifacts stay in /kaggle/working, download them from the file panel on the right.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU: Settings sidebar > Accelerator > GPU'
!test -d SubSight || git clone https://github.com/manocw/SubSight.git 2>&1 | tail -1
%cd /kaggle/working/SubSight
!git pull --ff-only 2>&1 | tail -1
!pip install -q -r requirements.txt transformers 2>&1 | tail -1

In [ ]:
# Data: Mini only (merged 647-pair set, skips Mini2/SSS to fit disk). Zip deleted after extract.
import socket, time, urllib.request, zipfile
from pathlib import Path

socket.setdefaulttimeout(60)
URL = 'https://zenodo.org/api/records/12666132/files/SubPipeMini.zip/content'
TOTAL = 6078303615
out = Path('data/SubPipeMini.zip')
out.parent.mkdir(exist_ok=True)

if not (out.exists() and out.stat().st_size >= 0.95 * TOTAL):
    for attempt in range(1, 9):
        have = out.stat().st_size if out.exists() else 0
        req = urllib.request.Request(URL, headers={'Range': f'bytes={have}-', 'User-Agent': 'SubSight-kaggle'})
        try:
            r = urllib.request.urlopen(req, timeout=60)
            mode = 'ab'
            if have and r.status == 200:
                mode, have = 'wb', 0
            print(f'downloading from {have / 1e9:.2f} GB (attempt {attempt})')
            with open(out, mode) as f:
                last = time.time()
                while True:
                    chunk = r.read(4 * 1024 * 1024)
                    if not chunk:
                        break
                    f.write(chunk)
                    f.flush()
                    if time.time() - last > 60:
                        done = f.tell()
                        print(f'  {done / 1e9:.2f}/{TOTAL / 1e9:.2f} GB ({100 * done / TOTAL:.0f}%)', flush=True)
                        last = time.time()
        except Exception as e:
            print(f'  stall: {e}, retrying...')
            time.sleep(10)
            continue
        if out.stat().st_size >= 0.95 * TOTAL:
            break
    assert out.stat().st_size >= 0.95 * TOTAL, 'download incomplete, rerun cell to resume'
print(f'zip ok ({out.stat().st_size / 1e9:.2f} GB)')

marker = Path('data/.extracted_Mini.ok')
if not marker.exists():
    print('extracting...')
    with zipfile.ZipFile(out) as f:
        print('zip sample:', f.namelist()[:3])
        f.extractall('data')
    marker.touch()
    out.unlink()
    print('zip deleted to save disk')
else:
    print('already extracted')

from src.dataset import find_pairs
roots = sorted(str(p) for p in Path('data').glob('**/Segmentation'))
print('segmentation dirs:', roots)
print('pairs:', len(find_pairs(roots[0])))

import yaml
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['data']['root'] = roots[0]
cfg['data']['split'] = 'chrono'
cfg['model']['architecture'] = 'segformer'
cfg['train']['checkpoint_dir'] = 'checkpoints_segformer'
yaml.safe_dump(cfg, open('configs/config.yaml', 'w'))
print('config:', cfg['data']['root'], '|', cfg['data']['split'], '|', cfg['model']['architecture'])

In [ ]:
!python -m src.train --config configs/config.yaml

In [ ]:
!python -m src.evaluate --checkpoint checkpoints_segformer/best.pth --num-images 6 --out outputs/eval_segformer.png
!python -m src.evaluate --checkpoint checkpoints_segformer/best.pth --test-split
!ls -la checkpoints_segformer/best.pth outputs/eval_segformer.png